In [ ]:
#
# Project:
#      PyTorch Dojo (https://github.com/wo3kie/ml-dojo)
#
# Author:
#      Lukasz Czerwinski (https://www.lukaszczerwinski.pl/)
#

In [ ]:
import torch as tr
import torch.nn as nn
import torch.autograd as autograd

import import_ipynb
import bce # type: ignore
import linear # type: ignore
import tanh # type: ignore
from per_lin_sig_bce import Per_Lin_Sig_BCE_Gradient # type: ignore
from common import repeat, test_checkgrad_2, test_compare_2, test_model_sgd_2 # type: ignore
from approx import approx # type: ignore

In [ ]:
class Per_Lin_Tanh_BCE_Autograd(nn.Module):
    """The version relying fully on PyTorch to handle `forward` and `backward` passes."""
    
    def __init__(self, in_features: int, out_features: int, W: tr.Tensor = None, b: tr.Tensor = None):
        super().__init__()

        self.linear =  nn.Linear(in_features, out_features)
        self.tanh = nn.Tanh()

        if W is not None:
            with tr.no_grad():
                self.linear.weight.copy_(W)

        if b is not None:
            with tr.no_grad():
                self.linear.bias.copy_(b)

    def forward(self, X):
        return self.model(X)
    
    def model(self, X):
        z = self.linear(X)
        return 0.5 * self.tanh(z) + 0.5      

    def loss(self, p, t):
        return nn.BCELoss(reduction='mean')(p, t)

    def predict(self, X):
        with tr.no_grad():
            return (self.model(X) > 0.5).float()

In [ ]:

class Per_Lin_Tanh_BCE_Backward(nn.Module):
    """
    The version where the `forward` and `backward` passes for each operation are written manually, 
    but PyTorch's autograd still orchestrates the overall gradient flow through the computational graph.
    """

    def __init__(self, in_features, out_features, w: tr.Tensor = None, b: tr.Tensor = None) -> None:
        super().__init__()

        #
        # [Linear Layer] + [BinaryCrossEntropyWithLogits = Tanh + BinaryCrossEntropy] 
        # is more numerically stable than [Linear] + [Tanh] + [BinaryCrossEntropy]
        #

        self.linear = linear.Linear(in_features, out_features, w, b)
        self.tanh = tanh.Tanh()

    def model(self, X):
        z = self.linear(X)
        return 0.5 * self.tanh(z) + 0.5
    
    def loss(self, p, t):
        return bce.BCE(reduction='mean')(p, t)

    def predict(self, X):
        with tr.no_grad():
            return (self.model(X) > 0.5).float()
        
    def forward(self, X):
        return self.model(X)

$$ \text{model function: linear} $$

$$ \mathbb{R} \times \mathbb{R} \to \mathbb{R}, \quad z(w, b) = xw + b $$

$$ 
\mathbb{R}^m \times \mathbb{R}^n \to \mathbb{R}^n, \quad z(\mathbf{w}, \mathbf{b}) = 
X\mathbf{w} + \mathbf{b}
$$

$$ \frac{\partial z}{\partial \mathbf{w}} = X $$

$$ \frac{\partial z}{\partial \mathbf{b}} = \mathbf{I}_{n} $$

$$ 
dz = 
\frac{\partial z}{\partial \mathbf{w}} d\mathbf{w} + \frac{\partial z}{\partial \mathbf{b}} d\mathbf{b} 
$$

$$ \text{activation function:} \quad \frac{1}{2} \tanh(z) + \frac{1}{2} $$

$$ \mathbb{R} \to (0, 1), \quad p(z) = \frac{1}{2} \frac{e^z - e^{-z}}{e^z + e^{-z}} + \frac{1}{2}$$

$$ 
\mathbb{R^n} \to (0, 1)^n, \quad p(\mathbf{z}) = 
(p(z_1), p(z_2), \dots, p(z_n)) 
$$

$$ \frac{dp}{dz} = \frac{1}{2} (1 - (2p -1)^2) $$

$$ \text{loss function: binary cross-entropy} $$

$$ 
(0, 1)^n \times \{0, 1\}^n \to \mathbb{R}, \quad L(\mathbf{p}, \mathbf{t}) = 
-\frac{1}{N} \sum \Big(t _i \ln(p_i) + (1 - t_i) \ln(1 - p_i) \Big) 
$$

$$
\frac{\partial L}{\partial p} = 
\frac{1}{N} \frac{p - t}{p \odot (1 - p)}
$$

$$ 
\frac{\partial L}{\partial z} = 
\frac{\partial L}{\partial p} \frac{\partial p}{\partial z} = 2 \frac{1}{N} (p - t) = 2 \frac{\partial \sigma}{\partial z}$$

$$ \text{Frobenius equality} $$

$$ \left \langle \frac{\partial F}{\partial \mathbf{w}}, d\mathbf{w} \right \rangle + 
\left \langle \frac{\partial F}{\partial \mathbf{b}}, d\mathbf{b} \right \rangle = 
\left \langle \frac{\partial F}{\partial L}, dL \right \rangle $$

$$ \\[2em] $$

$$ \left \langle \frac{\partial F}{\partial L}, dL \right \rangle = \frac{\partial F}{\partial L} \left \langle \frac{\partial L}{\partial \mathbf{z}}, d\mathbf{z} \right \rangle =  $$

$$ 
\frac{\partial F}{\partial L} \left \langle \frac{\partial L}{\partial \mathbf{z}}, \frac{\partial \mathbf{z}}{\partial \mathbf{w}} d\mathbf{w} + \frac{\partial \mathbf{z}}{\partial \mathbf{b}} d\mathbf{b} \right \rangle = 
$$

$$ 
\frac{\partial F}{\partial L} \left \langle \frac{\partial L}{\partial \mathbf{z}}, \frac{\partial \mathbf{z}}{\partial \mathbf{w}} d\mathbf{w} \right \rangle + 
\frac{\partial F}{\partial L} \left \langle \frac{\partial L}{\partial \mathbf{z}},\frac{\partial \mathbf{z}}{\partial \mathbf{b}} d\mathbf{b} \right \rangle = 
$$

$$ 
\frac{\partial F}{\partial L} \left \langle \Bigg( \frac{\partial \mathbf{z}}{\partial \mathbf{w}} \Bigg)^\top \frac{\partial L}{\partial \mathbf{z}}, d\mathbf{w} \right \rangle + 
\frac{\partial F}{\partial L} \left \langle \Bigg( \frac{\partial \mathbf{z}}{\partial \mathbf{b}} \Bigg)^\top \frac{\partial L}{\partial \mathbf{z}}, d\mathbf{b} \right \rangle = 
$$

$$ 
\left \langle \frac{\partial F}{\partial L} \Bigg( \frac{\partial \mathbf{z}}{\partial \mathbf{w}} \Bigg)^\top \frac{\partial L}{\partial \mathbf{z}}, d\mathbf{w} \right \rangle + 
\left \langle \frac{\partial F}{\partial L} \Bigg( \frac{\partial \mathbf{z}}{\partial \mathbf{b}} \Bigg)^\top \frac{\partial L}{\partial \mathbf{z}}, d\mathbf{b} \right \rangle \implies 
$$

$$ 
\begin{dcases}
\frac{\partial F}{\partial \mathbf{w}} = \frac{\partial F}{\partial L} \Bigg( \frac{\partial \mathbf{z}}{\partial \mathbf{w}} \Bigg)^\top \frac{\partial L}{\partial \mathbf{z}} = 1 \cdot X^\top \cdot \frac{\partial L}{\partial \mathbf{z}} \\
\\
\frac{\partial F}{\partial \mathbf{b}} = \frac{\partial F}{\partial L} \Bigg( \frac{\partial \mathbf{z}}{\partial \mathbf{b}} \Bigg)^\top \frac{\partial L}{\partial \mathbf{z}} = 1 \cdot I_n \cdot \frac{\partial L}{\partial \mathbf{z}}
\end{dcases}
$$


In [ ]:
class Per_Lin_Tanh_BCE_Gradient_Function(autograd.Function):
    @staticmethod
    def __model(X, w, b):
        z = linear.linear(X, w, b)
        return 0.5 * tanh.tanh(z) + 0.5
    
    @staticmethod
    def __loss(p, t):
        return bce.bce(p, t)

    @staticmethod
    def predict(X, w, b):
        with tr.no_grad():
            return (Per_Lin_Tanh_BCE_Gradient_Function.__model(X, w, b) >= 0.5).float()

    @staticmethod
    def forward(ctx, X, w, b, t):
        p = Per_Lin_Tanh_BCE_Gradient_Function.__model(X, w, b)
        L = Per_Lin_Tanh_BCE_Gradient_Function.__loss(p, t)

        ctx.save_for_backward(X, w, p, t)

        return L
    
    @staticmethod
    def backward(ctx, dF_dL):
        (X, w, p, t) = ctx.saved_tensors

        S = X.shape[0]  # Samples
        FI = X.shape[1] # Features In
        FO = w.shape[1] # Features Out
        N = S * FO
        
        dz_dw = X
        dL_dz = 2 * (1 / N) * (p - t)
        dF_dW = dF_dL * dz_dw.t() @ dL_dz
        dF_db = dF_dL * dL_dz.sum(dim=0)

        return (None, dF_dW, dF_db, None)
    

class Per_Lin_Tanh_BCE_Gradient(nn.Module):
    """
    The version exposing the exact mechanics of gradient computation and giving 
    full control over how the model participates in PyTorch's autograd system.
    """

    class _Linear:
        """Helper class to keep the model internal layout consistent across all variants."""

        def __init__(self, w, b):
            self.weight = w
            self.bias = b

    # This is helper for testing to indicate that the `forward` method expects both, `x` and `t`.
    CUSTOM_GRAD=True

    def __init__(self, in_features: int, out_features: int, W: tr.Tensor = None, b: tr.Tensor = None) -> None:
        super().__init__()

        # `weight` has shape `(out_features, in_features)` to be `nn.Linear` compatible
        if W is None:
            self.weight = nn.Parameter(0.1 * tr.randn(out_features, in_features, dtype=tr.float32))
        else:
            self.weight = nn.Parameter(W.clone().detach().requires_grad_(True))

        if b is None:
            self.bias = nn.Parameter(0.1 * tr.randn(out_features, dtype=tr.float32))
        else:
            self.bias = nn.Parameter(b.clone().detach().requires_grad_(True))

        self.linear = Per_Lin_Tanh_BCE_Gradient._Linear(self.weight, self.bias)

    def model(self, x):
        with tr.no_grad():
            return Per_Lin_Tanh_BCE_Gradient_Function.__model(x, self.weight.t(), self.bias)
    
    def loss(self, p, t):
        with tr.no_grad():
            return Per_Lin_Tanh_BCE_Gradient_Function.__loss(p, t)

    def predict(self, x):
        with tr.no_grad():
            return Per_Lin_Tanh_BCE_Gradient_Function.predict(x, self.weight.t(), self.bias)
        
    def forward(self, x, t):
        return Per_Lin_Tanh_BCE_Gradient_Function.apply(x, self.weight.t(), self.bias, t)

In [ ]:
def test_logical_fn(model, bool_fn, samples=100, epochs=500, lr=0.1):
    x = (tr.randn(samples, 2, dtype=tr.float32) > 0).float()
    t = bool_fn(x[:, 0], x[:, 1]).float().unsqueeze(1)
    return test_model_sgd_2(model, x, t, epochs, lr=lr)


if __name__ == "__main__":
    test_checkgrad_2(Per_Lin_Tanh_BCE_Gradient_Function, 1, 1, 1, prec=0.01)
    test_checkgrad_2(Per_Lin_Tanh_BCE_Gradient_Function, 2, 2, 2, prec=0.01)
    test_checkgrad_2(Per_Lin_Tanh_BCE_Gradient_Function, 3, 3, 3, prec=0.01)

    test_compare_2(Per_Lin_Tanh_BCE_Autograd, Per_Lin_Tanh_BCE_Backward, 1, 1, 1)
    test_compare_2(Per_Lin_Tanh_BCE_Autograd, Per_Lin_Tanh_BCE_Backward, 10, 1, 1)
    test_compare_2(Per_Lin_Tanh_BCE_Autograd, Per_Lin_Tanh_BCE_Backward, 10, 20, 1)
    test_compare_2(Per_Lin_Tanh_BCE_Autograd, Per_Lin_Tanh_BCE_Backward, 10, 20, 30)

    test_compare_2(Per_Lin_Tanh_BCE_Autograd, Per_Lin_Tanh_BCE_Gradient, 1, 1, 1)
    test_compare_2(Per_Lin_Tanh_BCE_Autograd, Per_Lin_Tanh_BCE_Gradient, 10, 1, 1)
    test_compare_2(Per_Lin_Tanh_BCE_Autograd, Per_Lin_Tanh_BCE_Gradient, 10, 20, 1)
    test_compare_2(Per_Lin_Tanh_BCE_Autograd, Per_Lin_Tanh_BCE_Gradient, 10, 20, 30)

    # Tanh keeps gradients alive much longer than sigmoid, so linear+tanh+BCE converges faster on logical functions.

    for epochs in [100, 200, 300, 400, 500]:
        print(f"{epochs}/OR/Tanh: {repeat(lambda: test_logical_fn(Per_Lin_Tanh_BCE_Gradient(2, 1), tr.logical_or, epochs=epochs)):.2f}")
        print(f"{epochs}/OR/Sig : {repeat(lambda: test_logical_fn(Per_Lin_Sig_BCE_Gradient(2, 1), tr.logical_or, epochs=epochs)):.2f}\n")

    # # The perceptron Linear + Tanh + BCE is still linear, and eventhough it converges faster than sigmoid, it still cannot learn the XOR function.

    for epochs in [500, 1000]:
        print(f"{epochs}/XOR/Tanh: {repeat(lambda: test_logical_fn(Per_Lin_Tanh_BCE_Gradient(2, 1), tr.logical_xor, epochs=epochs)):.2f}")
        print(f"{epochs}/XOR/Sig : {repeat(lambda: test_logical_fn(Per_Lin_Sig_BCE_Gradient(2, 1), tr.logical_xor, epochs=epochs)):.2f}\n")